# Decomposing Cliffords into transvections (π/4 Pauli exponents)

Every Clifford unitary can be written as an ordered product of **Clifford transvections** — the
`π/4` Pauli exponents $\exp\!\big(i\tfrac{\pi}{4} P_v\big)$. Conjugation by such an exponent acts on
Pauli operators as a **symplectic transvection**

$$
x \;\mapsto\; x + \langle x, v\rangle\, v,
$$

where $\langle\cdot,\cdot\rangle$ is the symplectic (commutation) form. `paulimer` exposes two
decompositions, following the transvection framework of
[arXiv:2102.11380](https://arxiv.org/abs/2102.11380) (Pllaha, Volanto & Tirkkonen,
*Decomposition of Clifford Gates*):

- [`CliffordUnitary.to_transvections`](../paulimer.pyi) — a greedy reduction that always returns a
  **linear** number of factors ($O(n)$),
- [`CliffordUnitary.to_transvections_minimal`](../paulimer.pyi) — the **strict minimum** number of
  factors.

Both reproduce the Clifford's **symplectic (conjugation) action** only; the Pauli-image signs and
the global phase are *not* preserved (the sign of a transvection does not change its symplectic
action).

In [1]:
import paulimer
from paulimer import CliffordUnitary, SparsePauli, DensePauli

## A single transvection

A `π/4` Pauli exponent *is* a Clifford transvection, so the simplest Cliffords decompose into a
single factor. The phase gate $S = \exp(-i\tfrac{\pi}{4} Z)$ and the Hadamard are both single
transvections (recall the returned sign is irrelevant to the symplectic action):

In [2]:
s_gate = CliffordUnitary.from_name("SqrtZ", [0], qubit_count=1)
hadamard = CliffordUnitary.from_name("Hadamard", [0], qubit_count=1)

print("S        ->", s_gate.to_transvections_minimal())
print("Hadamard ->", hadamard.to_transvections_minimal())

S        -> [Z]
Hadamard -> [-𝑖Y]


## Rebuilding a Clifford and checking the symplectic action

Applying the returned transvections in order with
[`left_mul_pauli_exp`](../paulimer.pyi) reconstructs the original **symplectic matrix**. We compare
`symplectic_matrix` (not the full signed tableau, since signs and global phase are not tracked by
this decomposition).

The minimal factor count is either $r$ or $r+1$, where the **residue rank**

$$
r \;=\; 2n - \dim \operatorname{Fix}(F)
$$

is the codimension of the space of Pauli operators fixed under conjugation. In `paulimer`,
$\dim\operatorname{Fix}(F)$ is the size of the Clifford's centralizer.

In [3]:
def residue_rank(clifford):
    return 2 * clifford.qubit_count - len(clifford.centralizer())


def rebuild(factors, qubit_count):
    rebuilt = CliffordUnitary.identity(qubit_count)
    for pauli in factors:
        rebuilt.left_mul_pauli_exp(pauli)
    return rebuilt


factors = s_gate.to_transvections_minimal()
rebuilt = rebuild(factors, s_gate.qubit_count)
print("residue rank r      =", residue_rank(s_gate))
print("number of factors   =", len(factors))
print("symplectic action ok:", rebuilt.symplectic_matrix == s_gate.symplectic_matrix)

residue rank r      = 1
number of factors   = 1
symplectic action ok: True


## Greedy versus minimal, and the $r+1$ case

For many Cliffords the greedy and minimal decompositions agree, but not always. The CNOT gate has
residue rank $r = 2$ yet needs $r + 1 = 3$ transvections: its symplectic action is *hyperbolic*
($\langle v, vF\rangle = 0$ for all $v$), which forces one extra factor.

In [4]:
cnot = CliffordUnitary.from_name("ControlledX", [0, 1], qubit_count=2)

greedy = cnot.to_transvections()
minimal = cnot.to_transvections_minimal()
print("residue rank r =", residue_rank(cnot))
print("greedy  :", greedy, " (", len(greedy), "factors )")
print("minimal :", minimal, " (", len(minimal), "factors )")
print("symplectic action ok:", rebuild(minimal, 2).symplectic_matrix == cnot.symplectic_matrix)

residue rank r = 2
greedy  : [Z, ZX, IX]  ( 3 factors )
minimal : [Z, ZX, IX]  ( 3 factors )
symplectic action ok: True


## A subtle case: non-hyperbolic maps that still need $r+1$

The 2021 paper claims that *every* non-hyperbolic Clifford decomposes into exactly $r$ transvections.
That is **not correct over $\mathbb{F}_2$**: some non-hyperbolic maps still require $r + 1$. The
smallest example already occurs on two qubits — the symplectic action built below (a product of the
transvections $X_0, X_1, X_0X_1, Z_0$) has residue rank $r = 3$, is non-hyperbolic, yet needs $4$
transvections. `to_transvections_minimal` returns the correct minimum. See
[`docs/transvection-minimality-correction.md`](../../../docs/transvection-minimality-correction.md)
for the full analysis and a machine-checked proof.

In [5]:
example = CliffordUnitary.identity(2)
for pauli in ["X0", "X1", "X0 X1", "Z0"]:
    example.left_mul_pauli_exp(SparsePauli(pauli))

minimal = example.to_transvections_minimal()
r = residue_rank(example)
print("residue rank r      =", r)
print("minimal factors     =", minimal, "(", len(minimal), "factors )")
print("needs r + 1          :", len(minimal) == r + 1)
print("symplectic action ok:", rebuild(minimal, 2).symplectic_matrix == example.symplectic_matrix)

residue rank r      = 3
minimal factors     = [IX, XX, -𝑖Y, X] ( 4 factors )
needs r + 1          : True
symplectic action ok: True


## Summary

- Clifford transvections are `π/4` Pauli exponents; `to_transvections` /
  `to_transvections_minimal` decompose any Clifford into them, reproducing its symplectic action
  with $O(n)$ factors.
- The minimal count is $r$ or $r + 1$, where $r = 2n - \dim\operatorname{Fix}(F)$.
- Only the symplectic action is reproduced — Pauli-image signs and the global phase are not.

### References

- T. Pllaha, K. Volanto, O. Tirkkonen, *Decomposition of Clifford Gates*, GLOBECOM 2021,
  [arXiv:2102.11380](https://arxiv.org/abs/2102.11380).
- [`docs/transvection-minimality-correction.md`](../../../docs/transvection-minimality-correction.md)
  — a correction to the paper's minimality claim, with a verified counterexample.